# CASMI 2026 - analogue + library retrieval

Builds every index from the competition data at runtime; no external assets.
Per molecule: mass-window pool, class-1 spectral cosine, class-2 analogue propagation, fused and deduplicated to 25 SMILES.

In [ ]:
import sys, time, pathlib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import glob, subprocess, sys, traceback
t_start = time.time()
try:
    import rdkit  # noqa: F401
except Exception:
    whl = glob.glob("/kaggle/input/**/rdkit-*.whl", recursive=True)
    if whl:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                        "--find-links", str(pathlib.Path(whl[0]).parent), "rdkit"],
                       check=False, capture_output=True)
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.*")
WORK = pathlib.Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

# The mount directory does not always match the slug's casing, so discover it.
_hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
print("input roots:", sorted(glob.glob("/kaggle/input/*"))[:20], flush=True)
if not _hits:
    raise SystemExit("train.parquet not found under /kaggle/input")
COMP = pathlib.Path(_hits[0]).parent
print("using competition dir:", COMP, flush=True)

# ---------------------------------------------------------------- chemistry
MASS = {"C":12.0,"H":1.00782503207,"N":14.0030740048,"O":15.9949146196,
        "P":30.97376163,"S":31.97207100,"F":18.99840322,"Cl":34.96885268,
        "Br":78.9183371,"I":126.904473,"Si":27.9769265325,"B":11.0093054,
        "Se":79.9165213,"Na":22.9897692809,"K":38.96370668}
PROTON = 1.007276466621
ELECTRON = 0.00054857990946
H2O = 2*MASS["H"] + MASS["O"]
HCOOH = MASS["C"] + 2*MASS["H"] + 2*MASS["O"]
DELTA = {"[M+H]+":PROTON, "[M+NH4]+":MASS["N"]+4*MASS["H"]-ELECTRON,
         "[M+Na]+":MASS["Na"]-ELECTRON, "[M+K]+":MASS["K"]-ELECTRON,
         "[M-H2O+H]+":-H2O+PROTON, "[M-2H2O+H]+":-2*H2O+PROTON,
         "[M-H]-":-PROTON, "[M-H2O-H]-":-H2O-PROTON, "[M+Cl]-":MASS["Cl"]+ELECTRON,
         "[M+CH2O2-H]-":HCOOH-PROTON, "[M+C2H4O2-H]-":MASS["C"]*2+MASS["H"]*4+MASS["O"]*2-PROTON}

def neutral_mass(mz, adduct):
    d = DELTA.get((adduct or "").strip())
    return None if d is None else mz - d

def formula_mass(f):
    if not isinstance(f, str) or not f: return None
    tot = 0.0
    import re
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", f):
        if not el: continue
        if el not in MASS: return None
        tot += MASS[el] * (int(n) if n else 1)
    return tot or None

def clean_peaks(mz, it, min_rel=0.01, max_peaks=128, prec=None, slack=2.0):
    mz = np.asarray(mz, dtype=np.float64); it = np.asarray(it, dtype=np.float64)
    n = min(len(mz), len(it)); mz, it = mz[:n], it[:n]
    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0) & (mz > 0)
    mz, it = mz[ok], it[ok]
    if len(mz) == 0: return mz, it
    if prec is not None:
        k = mz <= prec + slack
        if k.any(): mz, it = mz[k], it[k]
    it = it / it.max()
    k = it >= min_rel
    mz, it = mz[k], it[k]
    if len(mz) > max_peaks:
        t = np.argsort(it)[::-1][:max_peaks]; mz, it = mz[t], it[t]
    o = np.argsort(mz)
    return mz[o], it[o]

def cos(a_mz, a_it, b_mz, b_it, tol=0.01):
    if len(a_mz) == 0 or len(b_mz) == 0: return 0.0
    a = np.sqrt(a_it); b = np.sqrt(b_it)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0: return 0.0
    ob = np.argsort(b)[::-1]; used = np.zeros(len(b), bool); dot = 0.0
    for ia in np.argsort(a)[::-1]:
        cand = np.where(~used)[0]
        if len(cand) == 0: break
        d = np.abs(b_mz[cand] - a_mz[ia]); j = int(np.argmin(d))
        if d[j] <= tol:
            ib = int(cand[j]); used[ib] = True; dot += a[ia]*b[ib]
    return float(dot/(na*nb))

# ---------------------------------------------------------------- structures
# COCONUT matters: train is ~55% synthetic, so a train-only pool simply does not
# contain most natural-product answers.
_st = glob.glob("/kaggle/input/**/structures.parquet", recursive=True)
if _st:
    st = pd.read_parquet(_st[0])
    print("structure index from dataset:", _st[0], len(st), list(st.columns)[:8], flush=True)
else:
    tr = pq.read_table(COMP/"train.parquet",
                       columns=["inchikey14","normalized_smiles","molecular_formula"])
    st = (tr.to_pandas().dropna(subset=["inchikey14"])
            .drop_duplicates("inchikey14").reset_index(drop=True))
    st["neutral_mass"] = [formula_mass(f) for f in st["molecular_formula"]]
    print("WARNING: no structure dataset found, train-only pool", flush=True)

# one canonical column name regardless of source
if "smiles" not in st.columns:
    st = st.rename(columns={"normalized_smiles": "smiles"})
assert "smiles" in st.columns and "neutral_mass" in st.columns, list(st.columns)
print("structures usable:", len(st), flush=True)
DIAG = {"structures": int(len(st)), "src": str(_st[0]) if _st else "train-only"}
st = st.dropna(subset=["neutral_mass"]).sort_values("neutral_mass", kind="stable").reset_index(drop=True)
skey = st["inchikey14"].to_numpy(); smass = st["neutral_mass"].to_numpy()
ssmi = st["smiles"].to_numpy()
print(f"structures: {len(st):,}  ({time.time()-t_start:.0f}s)")

# ---------------------------------------------------------------- library
tb = pq.read_table(COMP/"train.parquet",
                   columns=["inchikey14","precursor_mz","molecular_formula",
                            "ms2_mzs","ms2_normalized_intensities"])
lm = tb.select(["inchikey14","precursor_mz","molecular_formula"]).to_pandas()
lm["neutral_mass"] = [formula_mass(f) for f in lm["molecular_formula"]]
ma = tb.column("ms2_mzs").combine_chunks(); ia = tb.column("ms2_normalized_intensities").combine_chunks()
moff, mflat = ma.offsets.to_numpy(), ma.values.to_numpy()
ioff, iflat = ia.offsets.to_numpy(), ia.values.to_numpy()
order = np.argsort(lm["neutral_mass"].fillna(1e9).to_numpy(), kind="stable")
lm = lm.iloc[order].reset_index(drop=True)
pm, pi, lens = [], [], np.zeros(len(lm), np.int64)
for j, i in enumerate(order):
    cm, cv = clean_peaks(mflat[moff[i]:moff[i+1]], iflat[ioff[i]:ioff[i+1]],
                         prec=float(lm["precursor_mz"].iat[j]))
    pm.append(cm.astype(np.float32)); pi.append(cv.astype(np.float32)); lens[j] = len(cm)
    if (j+1) % 500_000 == 0: print(f"  clean {j+1:,} {time.time()-t_start:.0f}s", flush=True)
lmz = np.concatenate(pm); lit = np.concatenate(pi)
loff = np.zeros(len(lens)+1, np.int64); np.cumsum(lens, out=loff[1:])
del pm, pi, tb, ma, ia, mflat, iflat
print(f"library: {len(lmz):,} peaks  ({time.time()-t_start:.0f}s)", flush=True)

BIN, MAXMZ = 0.05, 1200.0
from scipy import sparse
ra = np.repeat(np.arange(len(lens)), lens); keep = lmz <= MAXMZ
nb = int(np.ceil(MAXMZ/BIN))+1
X = sparse.csr_matrix((np.sqrt(lit[keep], dtype=np.float32),
                       (ra[keep], (lmz[keep]/BIN).astype(np.int32))), shape=(len(lens), nb), dtype=np.float32)
nrm = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())+1e-12
X = (sparse.diags((1.0/nrm).astype(np.float32)) @ X).tocsr()
print(f"sparse {X.shape} {X.nnz:,}  ({time.time()-t_start:.0f}s)", flush=True)

# ---------------------------------------------------------------- predict
test = pq.read_table(COMP/"test.parquet",
                     columns=["molecule_id","precursor_mz","adduct","ms2_mzs","ms2_normalized_intensities"]).to_pandas()
PPM, TOPK = 15.0, 40
ssmi_i = {k: v for k, v in zip(skey, ssmi)}
_fpc: dict = {}

def fp_of(smi):
    if smi not in _fpc:
        m = Chem.MolFromSmiles(str(smi)) if smi else None
        _fpc[smi] = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) if m is not None else None
    return _fpc[smi]

def zs(x):
    sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-9 else np.zeros_like(x)
li14 = lm["inchikey14"].to_numpy(); lmass = lm["neutral_mass"].to_numpy()
rows = []
_POOLS = []
for gi, (mid, g) in enumerate(test.groupby("molecule_id")):
    ms, specs = [], []
    for r in g.itertuples(index=False):
        m = neutral_mass(float(r.precursor_mz), str(r.adduct))
        if m is None: continue
        ms.append(m)
        cm, cv = clean_peaks(np.asarray(r.ms2_mzs), np.asarray(r.ms2_normalized_intensities),
                             prec=float(r.precursor_mz))
        if len(cm) >= 5: specs.append((cm, cv))
    if not ms or not specs:
        rows.append((mid, "C")); continue
    M = float(np.median(ms)); tol = M*PPM*1e-6
    lo = np.searchsorted(smass, M-tol, "left"); hi = np.searchsorted(smass, M+tol, "right")
    pool = np.arange(lo, hi)
    _POOLS.append(len(pool))
    if len(pool) == 0:
        rows.append((mid, "C")); continue
    pos = {k: i for i, k in enumerate(skey[pool])}
    cscore = np.zeros(len(pool))
    wlo = np.searchsorted(lmass, M - max(tol, M*2e-5), "left")
    whi = np.searchsorted(lmass, M + max(tol, M*2e-5), "right")
    for qm, qv in specs[:6]:
        for r in range(wlo, whi):
            s = cos(qm, qv, lmz[loff[r]:loff[r+1]], lit[loff[r]:loff[r+1]])
            j = pos.get(li14[r])
            if j is not None and s > cscore[j]: cscore[j] = s
    qm, qv = max(specs, key=lambda p: len(p[0]))
    q = sparse.csr_matrix((np.sqrt(qv, dtype=np.float32),
                           (np.zeros(len(qm), np.int32), (qm/BIN).astype(np.int32))), shape=(1, nb))
    q = q/(np.sqrt(q.multiply(q).sum())+1e-12)
    sims = np.asarray((X @ q.T).todense()).ravel()
    k = min(TOPK, len(sims)); top = np.argpartition(-sims, k-1)[:k]; top = top[sims[top] > 0]
    an = np.zeros(len(pool))
    if len(top):
        nf, nw = [], []
        for t in top:
            f = fp_of(ssmi_i.get(li14[t], ""))
            if f is not None:
                nf.append(f); nw.append(float(sims[t]))
        if nf:
            nw = np.asarray(nw); nw = nw / max(nw.sum(), 1e-9)
            for q, ci in enumerate(pool):
                f = fp_of(ssmi[ci])
                if f is not None:
                    an[q] = float(np.dot(DataStructs.BulkTanimotoSimilarity(f, nf), nw))
    sc = cscore + 0.35 * zs(an)
    o = np.argsort(-sc)[:25]
    seen, smis = set(), []
    for i in o:
        kk = skey[pool[i]]
        if kk in seen: continue
        seen.add(kk); smis.append(str(ssmi_i.get(kk, "")))
        if len(smis) == 25: break
    rows.append((mid, ";".join([s for s in smis if s]) or "C"))
    if (gi+1) % 50 == 0: print(f"  {gi+1}/{test['molecule_id'].nunique()} {time.time()-t_start:.0f}s", flush=True)

sub = pd.DataFrame(rows, columns=["molecule_id","smiles"])
sub = sub[sub["smiles"].astype(str).str.len() > 0]
sub.to_csv(WORK/"submission.csv", index=False)
DIAG["pool_median"] = float(np.median(_POOLS)) if _POOLS else -1
DIAG["rows"] = int(len(sub))
DIAG["secs"] = round(time.time()-t_start, 1)
import json as _json
(WORK/"diag.json").write_text(_json.dumps(DIAG, indent=1))
print(f"SUBMISSION {len(sub)} rows in {time.time()-t_start:.0f}s", flush=True)